# M3.S3 — Profiling and Performance Analysis
## From “it is slow” to an evidence-based diagnosis

This notebook accompanies **M3.S3 — Profiling and Performance Analysis**.

The objective is to move from:

> **“This code looks slow.”**

to:

> **“I measured it, located where time goes, formed a hypothesis, changed one thing, and proved whether it helped.”**

### Classroom method

> **PREDICT → MEASURE → LOCATE → EXPLAIN → CHANGE → REMEASURE**

### What we will do for real

- establish a fair measurement protocol;
- repeat wall-clock timings and inspect variability;
- profile one program with identifiable phases;
- compare **sampling** and **instrumentation**;
- distinguish a **hotspot** from a **bottleneck**;
- collect hardware-counter evidence when the cluster permits it;
- compare poor-locality and good-locality versions of the **same computation**;
- observe profiler overhead;
- measure MPI imbalance as real barrier waiting;
- inspect three evidence excerpts and decide the **next question**, not jump straight to a fix;
- finish with the workflow expected in **Practice 3**.

### Build

**M3S3-2026-09-26-v1**


# 0 — Environment and demo files

### Predict

Before running:

- Is the Jupyter/login node a good place to benchmark?
- Does the cluster allow access to Linux hardware performance counters?
- What should we do if `perf` exists but access to counters is restricted?

The notebook is designed so the core lesson still works even if hardware-counter access is restricted.


In [ ]:
import os
import platform
import re
import shutil
import statistics
import subprocess
import time
import urllib.request
from pathlib import Path

def show_rows(headers, rows):
    widths = [len(str(h)) for h in headers]
    text_rows = [[str(v) for v in row] for row in rows]
    for row in text_rows:
        widths = [max(w, len(v)) for w, v in zip(widths, row)]
    print(" | ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("-+-".join("-" * w for w in widths))
    for row in text_rows:
        print(" | ".join(v.ljust(w) for v, w in zip(row, widths)))

def run_command(command):
    print("$", command)
    p = subprocess.run(command, shell=True, executable="/bin/bash",
                       capture_output=True, text=True)
    if p.stdout.strip():
        print(p.stdout.strip())
    if p.stderr.strip():
        print("[stderr]")
        print(p.stderr.strip())
    print("return code:", p.returncode)
    print()
    return p

print("Host:", platform.node())
print("User:", os.environ.get("USER", "unknown"))
print("Visible CPUs:", os.cpu_count())
print("sbatch:", shutil.which("sbatch"))
print("gcc:", shutil.which("gcc"))
print("mpicc:", shutil.which("mpicc"))
print("perf:", shutil.which("perf"))
print("gprof:", shutil.which("gprof"))
print()

run_command("cat /proc/sys/kernel/perf_event_paranoid 2>/dev/null || true")

DEMO_NAMES = [
    "m3s3_profile_demo.c",
    "m3s3_mpi_wait.c",
    "m3s3_profile_lab.sbatch",
]

RAW_BASE = (
    "https://raw.githubusercontent.com/"
    "OscarDiez/hpc_course/main/session_demos/13_profiling_analysis/"
)

FILES_READY = True
for name in DEMO_NAMES:
    try:
        urllib.request.urlretrieve(RAW_BASE + name, name)
        print(f"{name:<28} <- GitHub")
    except Exception as exc:
        FILES_READY = False
        print(f"{name:<28} !! {exc}")

print("\nDemo files ready:", FILES_READY)


### Explain

Seeing `gcc`, `perf`, or hundreds of CPUs in Jupyter does **not** mean we should benchmark here.

For valid performance evidence we want:

> **same workload + same build + declared resources + repeated runs**

The real experiments therefore run through **Slurm** on a compute node.


# 1 — A valid measurement protocol

Before choosing a profiler, define the experiment.

For every comparison, record:

- **workload** — same input and problem size;
- **build** — same code version and compiler flags;
- **resources** — same node type, cores, ranks, GPUs;
- **measurement boundary** — what starts and stops the timer;
- **repetitions** — enough runs to see variability;
- **reported statistic** — median/mean plus variability;
- **one changed variable** — otherwise the comparison is ambiguous.

### Question

Which of these comparisons is fair?

**A** — same input, same node, same binary, before/after one loop-order change  
**B** — different input sizes and different core counts  
**C** — debug build before, `-O3` build after  
**D** — one run before and five runs after

<details>
<summary><strong>Reveal</strong></summary>

**A** is the clean comparison.

</details>


# 2 — The real profile-demo program

The program deliberately has identifiable phases:

```c
initialise(...)
compute_bad(...)   // or compute_good(...)
reduction_work(...)
other_work(...)
```

The key comparison uses the **same matrix** and the **same arithmetic**.

## Bad access order

```c
for (int j = 0; j < n; ++j)
    for (int i = 0; i < n; ++i)
        sum += A[i*n + j];
```

## Good access order

```c
for (int i = 0; i < n; ++i)
    for (int j = 0; j < n; ++j)
        sum += A[i*n + j];
```

Both compute the same sum.

### Predict

1. Which version should run faster in C?
2. Will a profiler tell us **where** time goes?
3. Will that automatically tell us **why**?


# 3 — Run the real profiling lab on the cluster

One Slurm job:

- builds the uninstrumented program;
- runs **5 baseline timings**;
- runs **5 improved timings**;
- builds and runs a `gprof`-instrumented version;
- attempts `perf stat` hardware counters;
- attempts `perf record` sampling;
- runs a real 4-rank MPI imbalance example.

The job is deliberately tolerant of restricted `perf` permissions: the rest of the experiment still completes.


In [ ]:
def clean_slurm_env():
    env = os.environ.copy()
    for key in ("SLURM_MEM_PER_CPU", "SLURM_MEM_PER_GPU", "SLURM_MEM_PER_NODE"):
        env.pop(key, None)
    return env

def submit_slurm(script):
    if shutil.which("sbatch") is None:
        print("sbatch is not available here — run this on SciTech JupyterHub.")
        return None
    p = subprocess.run(
        ["sbatch", "--parsable", script],
        capture_output=True,
        text=True,
        env=clean_slurm_env(),
    )
    if p.returncode != 0:
        print("Slurm submission failed:")
        print(p.stderr)
        return None
    job_id = p.stdout.strip().split(";")[0]
    print("Submitted Slurm job:", job_id)
    return job_id

def wait_for_job(job_id, timeout=900, poll=5):
    if not job_id or shutil.which("squeue") is None:
        return False
    start = time.time()
    while time.time() - start < timeout:
        p = subprocess.run(
            ["squeue", "-h", "-j", str(job_id), "-o", "%T"],
            capture_output=True, text=True
        )
        state = p.stdout.strip()
        if not state:
            print("Job", job_id, "has left the queue.")
            return True
        print("Job", job_id, "state:", state)
        time.sleep(poll)
    print("Timed out waiting. The job may still be queued/running.")
    return False

def read_output(job_id):
    path = Path(f"m3s3_profile-{job_id}.out")
    if not path.exists():
        print("Output file not found:", path)
        return ""
    return path.read_text(errors="replace")

def section(text, label):
    start = f"===== {label} ====="
    end = f"===== END {label} ====="
    i = text.find(start)
    j = text.find(end)
    if i < 0 or j < 0:
        return ""
    return text[i:j + len(end)]

def kv_lines(text, prefix):
    rows = []
    for line in text.splitlines():
        if not line.startswith(prefix):
            continue
        row = {}
        for token in line.split():
            if "=" in token:
                k, v = token.split("=", 1)
                row[k] = v
        rows.append(row)
    return rows

print("Slurm helpers ready.")


In [ ]:
M3S3_JOB_ID = None
M3S3_OUTPUT = ""

if FILES_READY and shutil.which("sbatch"):
    M3S3_JOB_ID = submit_slurm("m3s3_profile_lab.sbatch")
    if wait_for_job(M3S3_JOB_ID):
        time.sleep(1)
        M3S3_OUTPUT = read_output(M3S3_JOB_ID)
        print("\nM3S3 profiling lab completed.")
        print("PASS marker found:", "M3S3_PROFILE_LAB_RESULT=PASS" in M3S3_OUTPUT)
        print("\n" + section(M3S3_OUTPUT, "ENVIRONMENT"))
else:
    print("Run this notebook on SciTech JupyterHub to execute the real experiments.")


# 4 — Can you trust one timing?

The job ran the poor-locality version **five times**.

### Predict

- Will all five runs be identical?
- If one run is noticeably slower, should we report only that run?
- Would median be more robust than one arbitrary measurement?


In [ ]:
bad_totals = [float(r["seconds"]) for r in kv_lines(M3S3_OUTPUT, "PROFILE_TOTAL")
              if r.get("mode") == "bad"]

if bad_totals:
    rows = [(i+1, f"{t:.6f}") for i, t in enumerate(bad_totals[:5])]
    show_rows(["Run", "Wall time (s)"], rows)
    vals = bad_totals[:5]
    print()
    print(f"Median: {statistics.median(vals):.6f} s")
    print(f"Mean:   {statistics.mean(vals):.6f} s")
    print(f"Min:    {min(vals):.6f} s")
    print(f"Max:    {max(vals):.6f} s")
    print(f"Range:  {max(vals)-min(vals):.6f} s")
else:
    print("Run Section 3 first.")


### Explain

One timing is weak evidence.

A reasonable classroom protocol is:

> repeat → inspect variability → report a representative value

The exact statistic depends on the experiment, but the important habit is to make variability visible rather than hide it.


# 5 — Whole-program timing vs region timing

The program prints **phase timings** as well as total time.

That lets us ask two different questions:

- **How long does the whole application take?**
- **Which region contributes most to runtime?**

Run the next cell.


In [ ]:
rows = [r for r in kv_lines(M3S3_OUTPUT, "PROFILE_PHASE")
        if r.get("mode") == "bad"]

if rows:
    by_phase = {}
    for r in rows:
        by_phase.setdefault(r["phase"], []).append(float(r["seconds"]))
    table = []
    for phase, vals in by_phase.items():
        table.append((phase, f"{statistics.median(vals):.6f}"))
    table.sort(key=lambda x: float(x[1]), reverse=True)
    show_rows(["Phase", "Median region time (s)"], table)
else:
    print("Run Section 3 first.")


### Question

If `compute` dominates total runtime, what do we know?

We know **where a lot of time is spent**.

We do **not yet know why** it is slow.

> **Hotspot = location. Bottleneck = explanation of what limits progress.**


# 6 — Sampling vs instrumentation

## Sampling

Periodically asks:

> **Where is the program executing right now?**

Good for finding hotspots with relatively low overhead.

Example tool:

```bash
perf record -F 99 -g ./app
perf report
```

## Instrumentation

Adds measurement hooks so execution is explicitly recorded.

Example in this notebook:

```bash
gcc -pg ...
./app
gprof ...
```

Instrumentation can provide more detailed function/call information, but it can perturb the program more.

### Rule of thumb

> **Start with the least intrusive method that can answer the question.**


# 7 — Real instrumentation: find the hotspot with gprof

The Slurm job compiled a second binary with:

```bash
gcc -O3 -g -pg ...
```

and then ran `gprof`.

The output below is real profiler output from the compute node.


In [ ]:
gprof_text = section(M3S3_OUTPUT, "GPROF INSTRUMENTATION")
print(gprof_text if gprof_text else "Run Section 3 first.")


### Observe

Look for the function with the largest percentage of sampled time.

### What do we know now?

- which function dominates;
- approximately how the runtime is distributed.

### What do we still NOT know?

- whether the hotspot is compute-bound;
- memory-bound;
- branch-limited;
- synchronization-limited;
- or already close to optimal.

A hotspot is a **place to investigate**, not an automatic instruction to rewrite code.


# 8 — Real sampling with perf, if the cluster permits it

The same Slurm job attempts:

```bash
perf record -F 99 -g ./m3s3_profile_demo bad
perf report
```

Some HPC centres restrict access to `perf_event_open` for security or policy reasons.

That restriction is itself realistic HPC practice.

The next cell shows either:

- the real sampling report; or
- the real permission/error message from the node.


In [ ]:
perf_sampling = section(M3S3_OUTPUT, "PERF SAMPLING")
print(perf_sampling if perf_sampling else "Run Section 3 first.")


### Explain

If `perf` is restricted, do **not** invent counter values.

Use the evidence that is available:

- wall-clock timing;
- explicit phase timing;
- `gprof`/other allowed profiler;
- a recorded trace/profile captured on an environment where access is permitted.

> **A tool failing is not permission to fabricate measurements.**


# 9 — Hardware counters: ask a more specific “why?”

The job also attempts:

```bash
perf stat -e cycles,instructions,cache-references,cache-misses,branches,branch-misses
```

for both versions.

### Predict

For poor locality, which counter would you most want to compare?

- cache misses;
- branch misses;
- instruction count;
- all of the above, but interpreted in context?


In [ ]:
bad = section(M3S3_OUTPUT, "PERF STAT BAD")
good = section(M3S3_OUTPUT, "PERF STAT GOOD")

print("BAD ACCESS ORDER")
print(bad if bad else "No output")
print()
print("GOOD ACCESS ORDER")
print(good if good else "No output")


### Explain

Counters provide **clues, not conclusions**.

For example:

> high cache-miss evidence + poor spatial locality + large runtime

supports a memory-locality hypothesis more strongly than any one observation alone.

If the counters are unavailable, our hypothesis can still be tested by changing only the access order and remeasuring.


# 10 — Change ONE thing and remeasure

We now compare:

```text
BAD:  column-style traversal of a C row-major array
GOOD: row-style traversal of the same array
```

Same:

- matrix size;
- arithmetic;
- compiler;
- node allocation;
- number of runs.

Only the **access order** changes.

### Predict

If poor locality is the main limitation, what should happen to runtime?


In [ ]:
bad_vals = [float(r["seconds"]) for r in kv_lines(M3S3_OUTPUT, "PROFILE_TOTAL")
            if r.get("mode") == "bad"][:5]
good_vals = [float(r["seconds"]) for r in kv_lines(M3S3_OUTPUT, "PROFILE_TOTAL")
             if r.get("mode") == "good"][:5]

if bad_vals and good_vals:
    b = statistics.median(bad_vals)
    g = statistics.median(good_vals)
    show_rows(
        ["Version", "Median wall time (s)", "Relative"],
        [
            ("Bad locality", f"{b:.6f}", "1.00x"),
            ("Good locality", f"{g:.6f}", f"{b/g:.2f}x faster"),
        ],
    )
    print()
    print("Same computation result is checked by the program checksum.")
else:
    print("Run Section 3 first.")


### Evidence-led conclusion

If the good-locality version is substantially faster, that supports the hypothesis:

> **memory access order was limiting performance**

The reasoning chain is:

**baseline → hotspot → evidence → hypothesis → one change → remeasure**

That is stronger than:

> “This loop looked inefficient, so I rewrote it.”


# 11 — Measurement has a cost: the observer effect

Profiling changes the program.

The job runs:

- a normal optimized binary;
- a `-pg` instrumented binary.

### Question

Should the instrumented runtime be assumed identical to the baseline runtime?


In [ ]:
normal = bad_totals[:5] if 'bad_totals' in globals() else []
pg_rows = []
inside = section(M3S3_OUTPUT, "GPROF INSTRUMENTATION")
for line in inside.splitlines():
    if line.startswith("PROFILE_TOTAL"):
        row = {}
        for token in line.split():
            if "=" in token:
                k,v = token.split("=",1)
                row[k]=v
        if row.get("mode") == "bad":
            pg_rows.append(float(row["seconds"]))

if normal:
    print(f"Normal baseline median: {statistics.median(normal):.6f} s")
if pg_rows:
    print(f"Instrumented run:       {pg_rows[0]:.6f} s")
    print(f"Instrumented / baseline: {pg_rows[0]/statistics.median(normal):.3f}x")
else:
    print("The gprof run redirected program stdout in the batch script,")
    print("so use this section conceptually and the profiler output above.")


### Takeaway

More detailed observation can mean:

- more overhead;
- more data;
- altered timing behavior.

> **Start coarse. Add detail when the question requires it.**


# 12 — Parallel profiling: real MPI imbalance and barrier waiting

A separate real MPI program gives rank 3 more work:

```c
iters = (rank == size-1) ? 4 * base : base;
```

Then all ranks execute:

```c
MPI_Barrier(MPI_COMM_WORLD);
```

Each rank measures:

- useful compute time;
- time waiting in the barrier.

### Predict

Which ranks should show the most barrier waiting?


In [ ]:
rows = kv_lines(M3S3_OUTPUT, "MPI_TRACE")

if rows:
    table = []
    for r in sorted(rows, key=lambda x: int(x["rank"])):
        table.append((
            r["rank"],
            r["work_iters"],
            f'{float(r["compute"]):.6f}',
            f'{float(r["barrier_wait"]):.6f}',
        ))
    show_rows(["Rank", "Work iterations", "Compute (s)", "Barrier wait (s)"], table)
else:
    print("Run Section 3 first.")


### Explain

If rank 3 computes longest, the other ranks may arrive at the barrier earlier and wait.

The barrier is where waiting is **observed**.

The likely cause is **load imbalance before the barrier**.

> Optimising the already-fast waiting ranks does not fix the critical path.


# 13 — Performance detective: three evidence excerpts

Work in pairs.

For each excerpt, answer:

1. What do we know?
2. What do we **not** know?
3. What is the **next question or measurement**?

Do **not** jump directly to a code change.


In [ ]:
print("EXCERPT A — FUNCTION/PHASE PROFILE")
rows = [r for r in kv_lines(M3S3_OUTPUT, "PROFILE_PHASE") if r.get("mode")=="bad"]
if rows:
    by = {}
    for r in rows:
        by.setdefault(r["phase"], []).append(float(r["seconds"]))
    total = sum(statistics.median(v) for v in by.values())
    table = []
    for phase, vals in sorted(by.items(), key=lambda kv: statistics.median(kv[1]), reverse=True):
        med = statistics.median(vals)
        table.append((phase, f"{100*med/total:.1f}%"))
    show_rows(["Phase", "Approx. share"], table)
else:
    print("Run Section 3 first.")

print("\nEXCERPT B — HARDWARE EVIDENCE")
print(section(M3S3_OUTPUT, "PERF STAT BAD") or "Counters unavailable/restricted on this node.")

print("\nEXCERPT C — PARALLEL WAITING")
rows = kv_lines(M3S3_OUTPUT, "MPI_TRACE")
if rows:
    table = [(r["rank"], f'{float(r["compute"]):.4f}', f'{float(r["barrier_wait"]):.4f}')
             for r in sorted(rows,key=lambda x:int(x["rank"]))]
    show_rows(["Rank","Compute(s)","Barrier wait(s)"], table)


### Suggested reasoning

**A — Hotspot profile**

Next question:

> Why is the dominant compute phase slow?

Possible evidence: counters, access pattern, algorithmic work.

**B — Counter evidence**

Next question:

> Do the counter pattern and code access pattern support a memory-locality hypothesis?

Do not conclude from one counter in isolation.

**C — MPI waiting**

Next question:

> Why does one rank reach the barrier later?

Investigate work distribution before removing or tuning the barrier.

<details>
<summary><strong>Reveal after discussion</strong></summary>

The key learning outcome is not “choose the right fix”.

It is:

> **choose the next piece of evidence that reduces uncertainty.**

</details>


# 14 — Premature vs evidence-led optimisation

## Premature optimisation

```text
“This loop looks slow.”
        ↓
rewrite code
        ↓
hope
```

## Evidence-led optimisation

```text
Question
  ↓
Baseline
  ↓
Profile
  ↓
Hypothesis
  ↓
Change ONE thing
  ↓
Remeasure
  ↓
Keep / reject the change
```

### Question

If an optimisation makes one function 2× faster but total application runtime changes by only 1%, was it the right target?


# 15 — Practice 3 preview: prove an improvement

Your mission in Practice 3 is not simply:

> **make code faster**

It is:

> **prove that you identified a bottleneck and improved it using valid performance evidence**

Minimum evidence chain:

1. define the performance question;
2. establish a reproducible baseline;
3. locate a hotspot or symptom;
4. collect evidence relevant to the suspected bottleneck;
5. state an optimisation hypothesis;
6. change one thing;
7. rerun under comparable conditions;
8. report improvement **and limitations**.

### P3 pre-flight

Before the practice, make sure you can produce:

- one clean baseline run;
- the exact build command;
- the exact Slurm resource request;
- the input/problem size;
- enough repetitions to judge variability.


# 16 — Exit ticket

Answer briefly.

1. Why is one timing weak evidence?
2. What is the difference between a hotspot and a bottleneck?
3. When would you prefer sampling over heavy instrumentation?
4. What does a hardware counter provide: proof or a clue?
5. Why should we change one variable at a time?
6. If many MPI ranks wait at a barrier, where should we investigate first?
7. What is the final step after an optimisation?

<details>
<summary><strong>Concise answers</strong></summary>

1. Performance varies; one run does not reveal variability.
2. A hotspot is where time is spent; a bottleneck explains what limits progress.
3. When we need low-overhead localisation of CPU time/hotspots.
4. A clue that must be interpreted with other evidence.
5. So we can attribute the performance difference to that change.
6. The work before the barrier, especially the slow/critical rank.
7. Remeasure under comparable conditions and decide whether the evidence supports keeping the change.

</details>


# What you should leave with

- **Measure before profiling.**
- Define the performance question before choosing the tool.
- Repeated measurements are stronger evidence than one run.
- Sampling helps locate **where** CPU time goes.
- Instrumentation can add detail, but may add overhead.
- A **hotspot is not automatically a bottleneck**.
- Hardware counters provide clues about what the CPU experiences.
- Tracing/wait measurements help explain parallel behavior over time.
- Build a hypothesis from multiple pieces of evidence.
- Change **one thing**.
- Remeasure under comparable conditions.
- Keep an optimisation only when the evidence supports it.

The workflow is:

> **MEASURE → LOCATE → EXPLAIN → CHANGE → REMEASURE**

Next: **M3.S4 — Libraries, I/O and Reproducible Workflows**.
